# Résultats pipeline — CORTI 500

Features absolues · Isolation Forest + Autoencoder + PCA + Règles cliniques

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loader import load_json_file
from src.features import build_feature_matrix, preprocess, VISIT_CATEGORY_LABELS, NIHL_NOTCH_THRESHOLD_DB, compute_nihl_flag
from src.models.unsupervised import run_unsupervised_pipeline
from src.evaluate import (
    plot_audiogram,
    plot_top_anomalies,
    plot_patient_trajectory,
    summary_report,
    plot_flag_overlap,
    plot_anomaly_score_distribution,
)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
records = load_json_file('../CORTI_sample_audiograms_500.json')
df = pd.DataFrame(records)

print(f"Records : {len(df)}")
print(f"Patients uniques : {df['patient'].nunique()}")
print()
visit_counts = df['visit_category'].value_counts().sort_index()
for cat, count in visit_counts.items():
    label = VISIT_CATEGORY_LABELS.get(cat, '?')
    print(f"  {cat} {label:10s} : {count}")

In [ ]:
feature_df, feature_cols = build_feature_matrix(df)
X, scaler, imputer = preprocess(feature_df, fit=True)

print(f"Matrice features : {X.shape}")

scores_df, if_model, ae_model, _ = run_unsupervised_pipeline(
    X,
    contamination=0.05,
    ae_epochs=100,
    feature_df=feature_df,
)

summary_report(df, scores_df)

# Règles cliniques additionnelles
print("\n=== Règles cliniques ===")

# Règle NIHL : encoche 4 kHz ≥ 10 dB (formule directe Coles et al. 2000)
nihl_flag_4k = compute_nihl_flag(feature_df)
nihl_flag_4k.index = scores_df.index

# Règle Ménière : PTA basses fréquences corrigé > 25 dB
low_L = feature_df[["L_250", "L_500", "L_1000"]].mean(axis=1)
low_R = feature_df[["R_250", "R_500", "R_1000"]].mean(axis=1)
meniere_flag = ((low_L.fillna(0) > 25) | (low_R.fillna(0) > 25)).astype(int)
meniere_flag.index = scores_df.index

# Ajouter les flags au DataFrame
scores_df["nihl_flag_4k"] = nihl_flag_4k
scores_df["meniere_flag"] = meniere_flag

# Calculer anomaly_final (union ML + règles cliniques)
scores_df["anomaly_final"] = (
    (scores_df["anomaly_consensus"] == 1) |
    (scores_df["nihl_flag_4k"] == 1) |
    (scores_df["meniere_flag"] == 1)
).astype(int)

# Statistiques
print(f"{'anomaly_consensus (ML ≥2/3)':35s} : {scores_df['anomaly_consensus'].sum():>4} / {len(df)} ({100*scores_df['anomaly_consensus'].mean():.1f}%)")
print(f"{'nihl_flag_4k (encoche 4kHz >10dB)':35s} : {scores_df['nihl_flag_4k'].sum():>4} / {len(df)} ({100*scores_df['nihl_flag_4k'].mean():.1f}%)")
print(f"{'meniere_flag (PTA BF >25dB)':35s} : {scores_df['meniere_flag'].sum():>4} / {len(df)} ({100*scores_df['meniere_flag'].mean():.1f}%)")
print(f"{'anomaly_final (union)':35s} : {scores_df['anomaly_final'].sum():>4} / {len(df)} ({100*scores_df['anomaly_final'].mean():.1f}%)")

# Croisement consensus × NIHL
consensus = scores_df['anomaly_consensus'] == 1
nihl = scores_df['nihl_flag_4k'] == 1
both = (consensus & nihl).sum()
only_consensus = (consensus & ~nihl).sum()
only_nihl = (~consensus & nihl).sum()

print(f"\n=== Croisement Consensus × NIHL ===")
print(f"  Consensus ET NIHL        : {both:3d}")
print(f"  Consensus SANS NIHL      : {only_consensus:3d}")
print(f"  NIHL SANS consensus      : {only_nihl:3d}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_anomaly_score_distribution(scores_df['anomaly_score_if'], title='Isolation Forest', ax=axes[0])
plot_anomaly_score_distribution(scores_df['reconstruction_error'], title='Autoencoder', ax=axes[1])
plot_anomaly_score_distribution(scores_df['pca_reconstruction_error'], title='PCA baseline', ax=axes[2])
plt.tight_layout()
plt.show()

plot_flag_overlap(scores_df)
plt.show()

In [ ]:
GENDER_LABEL = {1: 'Homme', 2: 'Femme'}

# Utiliser anomaly_final (union ML + règles cliniques) au lieu de consensus seul
flagged_idx = scores_df[scores_df['anomaly_final'] == 1].index.tolist()
normal_idx  = scores_df[scores_df['anomaly_final'] == 0].index.tolist()
normal_df   = df.iloc[normal_idx]

print(f"{len(flagged_idx)} anomalies (union ML + règles)  |  {len(normal_idx)} cas normaux\n")

# Afficher les 8 premiers cas flaggés
for idx in flagged_idx[:8]:
    row    = df.iloc[idx]
    score  = scores_df.loc[idx, 'reconstruction_error']
    age    = row['age_at_visit']
    gender = GENDER_LABEL.get(row['gender'], '?')
    visit_label = VISIT_CATEGORY_LABELS.get(row['visit_category'], '?')

    # Infos encoche (méthode directe 4 kHz)
    notch_4k_L = feature_df.loc[idx, 'notch_4k_L']
    notch_4k_R = feature_df.loc[idx, 'notch_4k_R']
    notch_str = ""
    if not np.isnan(notch_4k_L) and notch_4k_L > 10:
        notch_str += f" | OG: encoche 4kHz={notch_4k_L:.0f}dB"
    if not np.isnan(notch_4k_R) and notch_4k_R > 10:
        notch_str += f" | OD: encoche 4kHz={notch_4k_R:.0f}dB"
    
    # Flags
    is_consensus = scores_df.loc[idx, 'anomaly_consensus'] == 1
    is_nihl = scores_df.loc[idx, 'nihl_flag_4k'] == 1
    is_meniere = scores_df.loc[idx, 'meniere_flag'] == 1
    
    flags_str = ""
    if is_consensus:
        flags_str += " [ML]"
    if is_nihl:
        flags_str += " [NIHL]"
    if is_meniere:
        flags_str += " [Ménière]"

    # Référence normale même sexe, âge proche
    pool = normal_df[normal_df['gender'] == row['gender']]
    if pool.empty:
        pool = normal_df
    if not np.isnan(age):
        ref_row = pool.iloc[(pool['age_at_visit'] - age).abs().argmin()]
    else:
        ref_row = pool.iloc[0]
    ref_score  = scores_df.loc[ref_row.name, 'reconstruction_error']
    ref_age    = ref_row['age_at_visit']
    ref_gender = GENDER_LABEL.get(ref_row['gender'], '?')

    age_str     = f"{age:.0f} ans"     if not np.isnan(age)     else "âge ?"
    ref_age_str = f"{ref_age:.0f} ans" if not np.isnan(ref_age) else "âge ?"

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_audiogram(
        row['dots_left'], row['dots_right'],
        title=f"ANOMALIE — {gender}, {age_str}, {visit_label} | err={score:.3f}{notch_str}{flags_str}",
        ax=axes[0],
    )
    plot_audiogram(
        ref_row['dots_left'], ref_row['dots_right'],
        title=f"RÉFÉRENCE NORMALE — {ref_gender}, {ref_age_str} | err={ref_score:.3f}",
        ax=axes[1],
    )
    plt.tight_layout()
    plt.show()

## Visualisation des cas NIHL détectés

Affichage des 10 cas avec les encoches 4 kHz les plus prononcées

In [ ]:
# Extraire les cas NIHL (encoche 4 kHz > 10 dB)
nihl_cases = scores_df[scores_df['nihl_flag_4k'] == 1].index.tolist()

# Calculer la profondeur maximale d'encoche pour chaque cas
notch_4k_max = feature_df[['nihl_notch_max_L', 'nihl_notch_max_R']].max(axis=1)

# Trier par profondeur décroissante et prendre les 10 premiers
nihl_sorted = notch_4k_max.loc[nihl_cases].sort_values(ascending=False)
top_nihl_idx = nihl_sorted.head(10).index.tolist()

print(f"Total cas NIHL détectés : {len(nihl_cases)}")
print(f"Affichage des 10 cas avec les encoches les plus prononcées\n")

for i, idx in enumerate(top_nihl_idx, 1):
    row = df.iloc[idx]
    age = row['age_at_visit']
    gender = GENDER_LABEL.get(row['gender'], '?')
    visit_label = VISIT_CATEGORY_LABELS.get(row['visit_category'], '?')
    
    # Encoches
    notch_4k_L = feature_df.loc[idx, 'notch_4k_L']
    notch_4k_R = feature_df.loc[idx, 'notch_4k_R']
    notch_max = max(notch_4k_L, notch_4k_R)
    
    # PTA pour contexte
    pta_L = feature_df.loc[idx, 'PTA_L']
    pta_R = feature_df.loc[idx, 'PTA_R']
    
    # Flags
    is_consensus = scores_df.loc[idx, 'anomaly_consensus'] == 1
    is_meniere = scores_df.loc[idx, 'meniere_flag'] == 1
    
    flags_str = " [NIHL]"
    if is_consensus:
        flags_str += " [ML]"
    if is_meniere:
        flags_str += " [Ménière]"
    
    age_str = f"{age:.0f} ans" if not np.isnan(age) else "âge ?"
    
    # Titre détaillé
    title = f"Cas NIHL #{i} (idx={idx}) — {gender}, {age_str}, {visit_label}\n"
    title += f"Encoche 4kHz : OG={notch_4k_L:.1f}dB, OD={notch_4k_R:.1f}dB (max={notch_max:.1f}dB)\n"
    title += f"PTA : OG={pta_L:.1f}dB, OD={pta_R:.1f}dB{flags_str}"
    
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_audiogram(
        row['dots_left'], row['dots_right'],
        title=title,
        ax=ax,
    )
    plt.tight_layout()
    plt.show()
    print()

## Statistiques des cas NIHL

In [ ]:
# Distribution des profondeurs d'encoche
nihl_depths = notch_4k_max.loc[nihl_cases]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme des profondeurs
axes[0].hist(nihl_depths, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(15, color='red', linestyle='--', linewidth=2, label='Seuil détection (15 dB)')
axes[0].axvline(nihl_depths.mean(), color='orange', linestyle='--', linewidth=2, label=f'Moyenne ({nihl_depths.mean():.1f} dB)')
axes[0].set_xlabel('Profondeur encoche 4 kHz (dB)')
axes[0].set_ylabel('Nombre de cas')
axes[0].set_title(f'Distribution des encoches NIHL (n={len(nihl_cases)})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Répartition par tranches
bins = [15, 20, 25, 30, 50]
labels = ['15-20 dB\n(léger)', '20-25 dB\n(modéré)', '25-30 dB\n(marqué)', '>30 dB\n(sévère)']
nihl_binned = pd.cut(nihl_depths, bins=bins, labels=labels, include_lowest=True)
counts = nihl_binned.value_counts().sort_index()

bars = axes[1].bar(range(len(counts)), counts.values, color=['lightblue', 'steelblue', 'orange', 'tomato'], 
                   edgecolor='white', alpha=0.8)
axes[1].set_xticks(range(len(counts)))
axes[1].set_xticklabels(counts.index, fontsize=9)
axes[1].set_ylabel('Nombre de cas')
axes[1].set_title('Répartition par sévérité')
axes[1].grid(True, alpha=0.3, axis='y')

# Annotations
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f'{val}\n({100*val/len(nihl_cases):.0f}%)', 
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Statistiques textuelles
print("\n=== Statistiques NIHL ===")
print(f"Nombre total de cas NIHL      : {len(nihl_cases)} / {len(df)} ({100*len(nihl_cases)/len(df):.1f}%)")
print(f"Profondeur moyenne            : {nihl_depths.mean():.1f} dB")
print(f"Profondeur médiane            : {nihl_depths.median():.1f} dB")
print(f"Profondeur min/max            : {nihl_depths.min():.1f} / {nihl_depths.max():.1f} dB")
print(f"\nCas NIHL détectés par ML      : {(scores_df.loc[nihl_cases, 'anomaly_consensus'] == 1).sum()} / {len(nihl_cases)} ({100*(scores_df.loc[nihl_cases, 'anomaly_consensus'] == 1).sum()/len(nihl_cases):.1f}%)")
print(f"Cas NIHL non détectés par ML  : {(scores_df.loc[nihl_cases, 'anomaly_consensus'] == 0).sum()} / {len(nihl_cases)} ({100*(scores_df.loc[nihl_cases, 'anomaly_consensus'] == 0).sum()/len(nihl_cases):.1f}%)")

## Comparaison : Anomalies (ML + NIHL) vs Normaux

Les cas NIHL sont triés par profondeur d'encoche décroissante.
- **NIHL seul** : détecté par la règle clinique mais pas par le ML
- **NIHL + ML** : détecté par les deux

In [ ]:
N_SHOW = 8  # nombre de cas à afficher

# Fusionner NIHL seul + NIHL+ML, triés par profondeur décroissante
nihl_idx       = scores_df[scores_df['nihl_flag_4k'] == 1].index.tolist()
nihl_only_idx  = [i for i in nihl_idx if scores_df.loc[i, 'anomaly_consensus'] == 0]
nihl_ml_idx    = [i for i in nihl_idx if scores_df.loc[i, 'anomaly_consensus'] == 1]
normal_idx_all = scores_df[scores_df['anomaly_final'] == 0].index.tolist()
normal_df_pool = df.iloc[normal_idx_all]

nihl_notch_max = feature_df[['nihl_notch_max_L', 'nihl_notch_max_R']].max(axis=1)
all_nihl_sorted = sorted(nihl_idx, key=lambda i: nihl_notch_max.loc[i], reverse=True)

print(f"NIHL seul (non ML) : {len(nihl_only_idx)} | NIHL + ML : {len(nihl_ml_idx)}")
print(f"Affichage des {N_SHOW} encoches les plus profondes vs normaux appariés\n")


def _notch_info(idx):
    best_freq, best_depth = '4kHz', 0.0
    for freq_label, col_L, col_R in [
        ('3kHz', 'notch_3k_L', 'notch_3k_R'),
        ('4kHz', 'notch_4k_L', 'notch_4k_R'),
        ('6kHz', 'notch_6k_L', 'notch_6k_R'),
    ]:
        depth = max(feature_df.loc[idx, col_L], feature_df.loc[idx, col_R])
        if depth > best_depth:
            best_depth, best_freq = depth, freq_label
    return best_freq, best_depth


# Construire les paires (anomalie, normal apparié)
pairs = []
for idx in all_nihl_sorted[:N_SHOW]:
    row = df.iloc[idx]
    age = row['age_at_visit']
    pool = normal_df_pool[normal_df_pool['gender'] == row['gender']]
    if pool.empty:
        pool = normal_df_pool
    ref_row = pool.iloc[(pool['age_at_visit'] - age).abs().argmin()] if not np.isnan(age) else pool.iloc[0]
    pairs.append((idx, ref_row.name))

# Figure : N lignes × 2 colonnes (gauche = anomalie, droite = normal)
fig, axes = plt.subplots(N_SHOW, 2, figsize=(14, 5 * N_SHOW))

for row_i, (anom_idx, norm_idx) in enumerate(pairs):
    anom_row = df.iloc[anom_idx]
    norm_row = df.iloc[norm_idx]

    age        = anom_row['age_at_visit']
    gender     = GENDER_LABEL.get(anom_row['gender'], '?')
    freq_label, depth = _notch_info(anom_idx)
    ae_err     = scores_df.loc[anom_idx, 'reconstruction_error']
    is_ml      = scores_df.loc[anom_idx, 'anomaly_consensus'] == 1
    tag        = '[NIHL+ML]' if is_ml else '[NIHL]'

    ref_age    = norm_row['age_at_visit']
    ref_gender = GENDER_LABEL.get(norm_row['gender'], '?')
    ref_err    = scores_df.loc[norm_idx, 'reconstruction_error']

    age_str     = f"{age:.0f} ans"     if not np.isnan(age)     else '?'
    ref_age_str = f"{ref_age:.0f} ans" if not np.isnan(ref_age) else '?'

    plot_audiogram(
        anom_row['dots_left'], anom_row['dots_right'],
        title=f"{tag} {gender}, {age_str} | encoche {freq_label}: {depth:.0f} dB | err={ae_err:.3f}",
        ax=axes[row_i, 0],
    )
    plot_audiogram(
        norm_row['dots_left'], norm_row['dots_right'],
        title=f"NORMAL {ref_gender}, {ref_age_str} | err={ref_err:.3f}",
        ax=axes[row_i, 1],
    )

axes[0, 0].set_title(axes[0, 0].get_title(), fontweight='bold', color='firebrick')
axes[0, 1].set_title(axes[0, 1].get_title(), fontweight='bold', color='steelblue')

fig.suptitle(
    f"Anomalies NIHL (encoche ≥ {NIHL_NOTCH_THRESHOLD_DB:.0f} dB, V-shape) vs Normaux appariés",
    fontsize=14, fontweight='bold', y=1.001,
)
plt.tight_layout()
plt.show()
